# Initial Importing

In [88]:

import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import utils
import numpy as np

closure_filepath = "/Users/hudson/Desktop/road_project/facebook/data/manual_closure_or_reopening_times_finished.xlsx"
closure_df = pd.read_excel(closure_filepath)
weather_filepath = "/Users/hudson/Desktop/road_project/weather/historical_weather_date.csv"
weather_df = pd.read_csv(weather_filepath)
weather_df = weather_df.drop(columns=["Unnamed: 0"], errors="ignore")
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce")
print(weather_df["date"].head())
print(weather_df["date"].dtype)
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.tz_localize(None)


0   2017-02-21 00:00:00+00:00
1   2017-02-21 01:00:00+00:00
2   2017-02-21 02:00:00+00:00
3   2017-02-21 03:00:00+00:00
4   2017-02-21 04:00:00+00:00
Name: date, dtype: datetime64[us, UTC]
datetime64[us, UTC]


In [89]:
intervals = utils.build_closure_intervals(
    closure_df,
    closure_col="timestamp",
    reopen_col="time_reopened"
)

weather_labeled = utils.apply_closure_to_weather(
    weather_df,
    intervals,
    weather_time_col="date"
)

## **Testing Whether it Works as Intended**

In [90]:
missing_example = intervals.loc[intervals["has_reopening_time"] == False].iloc[0]
missing_example
test_time = missing_example["closure_start"].floor("h")
print(test_time)
weather_labeled.loc[
    (weather_labeled["date"] >= test_time) &
    (weather_labeled["date"] <= test_time + pd.Timedelta(hours=23)),
    ["date", "closure"]
]

2017-02-21 21:00:00


,date,closure
21,2017-02-21 21:00:00,1.0
22,2017-02-21 22:00:00,1.0
23,2017-02-21 23:00:00,1.0
24,2017-02-22 00:00:00,NaN
25,2017-02-22 01:00:00,NaN
26,2017-02-22 02:00:00,NaN
27,2017-02-22 03:00:00,NaN
28,2017-02-22 04:00:00,NaN
29,2017-02-22 05:00:00,NaN
30,2017-02-22 06:00:00,NaN


In [91]:
weather_closure = weather_labeled.copy().sort_values("date").reset_index(drop=True)

weather_closure["closure_prev"] = weather_closure["closure"].shift(1)
weather_closure["closure_start"] = (
    (weather_closure["closure"] == 1) &
    (weather_closure["closure_prev"] != 1)
).astype(int)

print("Closure rows:", (weather_closure["closure"] == 1).sum())
print("Closure start events:", weather_closure["closure_start"].sum())

Closure rows: 1285
Closure start events: 374


In [92]:
print("Total closure starts:", weather_closure["closure_start"].sum())
print("Known closure rows:", (weather_closure["closure"] == 1).sum())
print("Ambiguous rows:", weather_closure["closure"].isna().sum())

Total closure starts: 374
Known closure rows: 1285
Ambiguous rows: 6144


In [93]:
df = weather_closure.copy()

# keep datetime clean
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# build winter season label
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

df["winter"] = np.where(
    df["month"] >= 10,
    df["year"].astype(str) + "-" + (df["year"] + 1).astype(str),
    (df["year"] - 1).astype(str) + "-" + df["year"].astype(str)
)

# if closure_start is not already there
df = df.sort_values("date").reset_index(drop=True)
df["closure_prev"] = df["closure"].shift(1)
df["closure_start"] = (
    (df["closure"] == 1) &
    (df["closure_prev"] != 1)
).astype(int)

# confirmed-status-only dataset
clean_df = df.dropna(subset=["closure"]).copy()

print("Clean rows:", len(clean_df))
print("Open rows:", (clean_df["closure"] == 0).sum())
print("Closed rows:", (clean_df["closure"] == 1).sum())
print("Closure rate:", (clean_df["closure"] == 1).mean())

print("\nClosure starts by winter:")
print(
    clean_df.groupby("winter")["closure_start"]
    .sum()
    .sort_index()
)

Clean rows: 73032
Open rows: 71747
Closed rows: 1285
Closure rate: 0.017595026837550664

Closure starts by winter:
winter
2016-2017    37
2017-2018    47
2018-2019    68
2019-2020    49
2020-2021    24
2021-2022    46
2022-2023    51
2023-2024    20
2024-2025    22
2025-2026    10
Name: closure_start, dtype: int64


In [94]:
forecast_df = clean_df.copy()
forecast_df = utils.make_future_closure_target(forecast_df, horizon_hours=24)

print(forecast_df["will_close_in_24h"].value_counts())
print(forecast_df["will_close_in_24h"].mean())

will_close_in_24h
0    67568
1     5464
Name: count, dtype: int64
0.07481651878628547


In [95]:
forecast_df = clean_df.copy()
forecast_df = utils.make_future_closure_target(forecast_df, horizon_hours=6)
forecast_df = utils.make_future_closure_target(forecast_df, horizon_hours=12)
forecast_df = utils.make_future_closure_target(forecast_df, horizon_hours=24)

for col in ["will_close_in_6h", "will_close_in_12h", "will_close_in_24h"]:
    print(col)
    print(forecast_df[col].value_counts())
    print("rate:", forecast_df[col].mean())
    print()

will_close_in_6h
will_close_in_6h
0    71594
1     1438
Name: count, dtype: int64
rate: 0.019689998904589768

will_close_in_12h
will_close_in_12h
0    70212
1     2820
Name: count, dtype: int64
rate: 0.038613210647387444

will_close_in_24h
will_close_in_24h
0    67568
1     5464
Name: count, dtype: int64
rate: 0.07481651878628547



In [96]:
print("len(clean_df):", len(clean_df))
print("len(forecast_df):", len(forecast_df))

print("closure_start sum in clean_df:", clean_df["closure_start"].sum())
print("closure_start sum in forecast_df:", forecast_df["closure_start"].sum())

print("duplicate dates in clean_df:", clean_df["date"].duplicated().sum())
print("duplicate dates in forecast_df:", forecast_df["date"].duplicated().sum())

len(clean_df): 73032
len(forecast_df): 73032
closure_start sum in clean_df: 374
closure_start sum in forecast_df: 374
duplicate dates in clean_df: 0
duplicate dates in forecast_df: 0


In [98]:
len(weather_closure)

79176